# Practice: Custom Functions on Real-World Data

This notebook:
1. Defines several reusable **functions** for loading, cleaning, summarizing, and plotting data.
2. Applies them to publicly available datasets:
   - **World Bank API** — GDP per capita & life expectancy
   - **Our World in Data CSV** — CO2 emissions
3. Demonstrates merging and comparing indicators across countries.

> Goal: build intuition by writing your own helpers instead of re-typing `pd.read_csv` + filtering boilerplate every time.

## 0. Setup

In [ ]:
import io
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 1. Reusable Functions

We define helpers once and reuse them across every analysis below.

### 1.1 `download_worldbank` — pull any indicator for any country list

In [ ]:
def download_worldbank(indicator: str, countries: list, year_start: int, year_end: int) -> pd.DataFrame:
    """Download a World Bank indicator as a tidy DataFrame.

    Parameters
    ----------
    indicator : World Bank indicator code, e.g. 'NY.GDP.PCAP.CD'
    countries : list of ISO3 codes, e.g. ['USA','CHN','DEU']
    year_start, year_end : inclusive year range

    Returns
    -------
    DataFrame with columns: country, iso3, year, value
    """
    codes = ";".join(countries)
    url = (f"https://api.worldbank.org/v2/country/{codes}"
           f"/indicator/{indicator}?date={year_start}:{year_end}&per_page=30000")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    raw = pd.read_xml(io.BytesIO(r.content))
    tidy = (raw[["country", "countryiso3code", "date", "value"]]
            .rename(columns={"countryiso3code": "iso3", "date": "year"})
            .astype({"year": int})
            .sort_values(["iso3", "year"])
            .reset_index(drop=True))
    return tidy

### 1.2 `load_csv_url` — cache a CSV download from any URL

In [ ]:
def load_csv_url(url: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download a CSV directly into a DataFrame. Extra kwargs forwarded to pd.read_csv."""
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), **read_csv_kwargs)

### 1.3 `summarize` — quick profile of a DataFrame

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Return a per-column summary: dtype, n_missing, n_unique, sample value."""
    return pd.DataFrame({
        "dtype":     df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique":  df.nunique(dropna=True),
        "sample":    [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

### 1.4 `filter_countries_years` — pick subset by country list and year range

In [ ]:
def filter_countries_years(df: pd.DataFrame, iso_col: str, year_col: str,
                            countries: list, year_start: int, year_end: int) -> pd.DataFrame:
    mask = df[iso_col].isin(countries) & df[year_col].between(year_start, year_end)
    return df.loc[mask].copy()

### 1.5 `plot_timeseries` — line chart, one line per country

In [ ]:
def plot_timeseries(df: pd.DataFrame, x: str, y: str, group: str,
                    title: str = "", ylabel: str = ""):
    fig, ax = plt.subplots(figsize=(9, 5))
    for key, sub in df.groupby(group):
        ax.plot(sub[x], sub[y], marker="o", markersize=3, label=key)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel(ylabel or y)
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return ax

## 2. Dataset A — GDP per capita (World Bank API)

Indicator `NY.GDP.PCAP.CD` = GDP per capita, current US$.

In [ ]:
COUNTRIES = ["USA", "CHN", "DEU", "JPN", "IND", "BRA"]

gdp = download_worldbank("NY.GDP.PCAP.CD", COUNTRIES, 1990, 2023)
gdp.head()

In [ ]:
summarize(gdp)

In [ ]:
plot_timeseries(gdp.dropna(subset=["value"]), x="year", y="value", group="country",
                title="GDP per capita (current US$), 1990-2023", ylabel="USD");

## 3. Dataset B — Life expectancy (World Bank API)

Indicator `SP.DYN.LE00.IN` = Life expectancy at birth, total (years).

In [ ]:
life = download_worldbank("SP.DYN.LE00.IN", COUNTRIES, 1990, 2022)
life.head()

In [ ]:
plot_timeseries(life.dropna(subset=["value"]), x="year", y="value", group="country",
                title="Life expectancy at birth, 1990-2022", ylabel="Years");

## 4. Dataset C — CO2 emissions (Our World in Data)

Public CSV at `https://github.com/owid/co2-data/raw/master/owid-co2-data.csv`.
We load it, then reuse `filter_countries_years` to slice.

In [ ]:
OWID_URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"
co2_raw = load_csv_url(OWID_URL, usecols=["iso_code", "country", "year", "co2_per_capita"])
co2_raw.head()

In [ ]:
co2 = filter_countries_years(co2_raw, iso_col="iso_code", year_col="year",
                              countries=COUNTRIES, year_start=1990, year_end=2022)
co2 = co2.rename(columns={"iso_code": "iso3", "co2_per_capita": "co2_pc"})
co2.head()

In [ ]:
plot_timeseries(co2.dropna(subset=["co2_pc"]), x="year", y="co2_pc", group="country",
                title="CO2 emissions per capita (tonnes), 1990-2022", ylabel="t CO2/person");

## 5. Merge all three indicators

We end up with one tidy panel: `(country, year, gdp_pc, life_exp, co2_pc)`.

In [ ]:
gdp_ren  = gdp.rename(columns={"value": "gdp_pc"})[["iso3", "country", "year", "gdp_pc"]]
life_ren = life.rename(columns={"value": "life_exp"})[["iso3", "year", "life_exp"]]
co2_ren  = co2[["iso3", "year", "co2_pc"]]

panel = (gdp_ren
         .merge(life_ren, on=["iso3", "year"], how="left")
         .merge(co2_ren,  on=["iso3", "year"], how="left"))
panel.head()

In [ ]:
summarize(panel)

## 6. Quick analysis using the panel

Does higher GDP per capita go with longer life expectancy? (Use the latest year available per country.)

In [ ]:
latest = (panel.dropna(subset=["gdp_pc", "life_exp"])
                 .sort_values("year")
                 .groupby("iso3", as_index=False)
                 .tail(1))
latest[["country", "year", "gdp_pc", "life_exp", "co2_pc"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(latest["gdp_pc"], latest["life_exp"], s=60)
for _, row in latest.iterrows():
    ax.annotate(row["iso3"], (row["gdp_pc"], row["life_exp"]),
                xytext=(5, 5), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("GDP per capita (USD, log scale)")
ax.set_ylabel("Life expectancy (years)")
ax.set_title("GDP vs life expectancy — latest year")
ax.grid(alpha=0.3)
plt.tight_layout();

## 7. Your turn

Try these extensions — all reuse the functions above, no new boilerplate:

1. Add more countries to `COUNTRIES` (e.g. `KOR`, `NGA`, `ZAF`).
2. Call `download_worldbank` with another indicator, e.g. `SE.ADT.LITR.ZS` (adult literacy) or `EN.ATM.CO2E.PC`.
3. Write a new function `growth_rate(df, value_col)` that adds a yearly % change column, then plot it.
4. Use `load_csv_url` on any other public CSV (e.g. from data.gov, OWID, Kaggle public).

## 8. AI Policy — cumulative legislation by country

Data source: **Stanford AI Index**, republished by OWID.
Slug verified: `cumulative-number-artificial-intelligence-bills-passed`.

> Note: this grapher is a **cross-section** — one row per country, value = cumulative
> AI-related bills passed 2016-2023. It is not a time series, so we will not
> use `plot_timeseries` here. Instead we rank, distribute, and merge with GDP.

In [ ]:
POLICY_URL = ("https://ourworldindata.org/grapher/"
              "cumulative-number-artificial-intelligence-bills-passed.csv")
bills = load_csv_url(POLICY_URL)
print("shape:", bills.shape, "years:", bills["Year"].unique())
bills.head()

In [ ]:
summarize(bills)

### 8.1 Rename the long value column

In [ ]:
value_col = [c for c in bills.columns if c not in ("Entity","Code","Year")][0]
print("original value column:\n ", value_col)

policy = (bills.rename(columns={"Entity":"country","Code":"iso3",
                                 "Year":"year", value_col:"cum_bills"})
                .dropna(subset=["iso3"]))
policy.head()

### 8.2 Top 15 countries by cumulative AI bills

In [ ]:
top15 = policy.nlargest(15, "cum_bills")[["country","iso3","cum_bills"]]
top15

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top15["country"][::-1], top15["cum_bills"][::-1])
ax.set_xlabel("Cumulative AI bills passed (2016-2023)")
ax.set_title("Top 15 legislators on AI")
plt.tight_layout();

### 8.3 Distribution — how many countries legislate at all?

In [ ]:
bins = [-0.5, 0.5, 1.5, 3.5, 10.5, 100]
labels = ["0 bills", "1 bill", "2-3", "4-10", "10+"]
buckets = pd.cut(policy["cum_bills"], bins=bins, labels=labels)
dist = buckets.value_counts().reindex(labels)
print(dist)

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(dist.index.astype(str), dist.values)
ax.set_ylabel("Number of countries")
ax.set_title("How many AI bills have countries passed?")
plt.tight_layout();

### 8.4 Merge with GDP per capita

Do richer countries legislate more on AI? Join with the `gdp` panel from Section 2 —
take each country's 2023 GDP per capita.

In [ ]:
gdp_2023 = (gdp.rename(columns={"value":"gdp_pc"})
                .query("year == 2023")[["iso3","gdp_pc"]])

# gdp panel only covers our 6 country list; for a richer scatter, re-download more countries
BIG_LIST = policy["iso3"].dropna().unique().tolist()
gdp_all = download_worldbank("NY.GDP.PCAP.CD", BIG_LIST, 2022, 2023)
gdp_latest = (gdp_all.dropna(subset=["value"])
                     .sort_values("year").groupby("iso3", as_index=False).tail(1)
                     .rename(columns={"value":"gdp_pc"})[["iso3","gdp_pc"]])

merged = policy.merge(gdp_latest, on="iso3", how="inner")
print("merged shape:", merged.shape)
merged.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(merged["gdp_pc"], merged["cum_bills"], alpha=0.6)

# annotate the most prolific legislators
for _, row in merged.nlargest(8, "cum_bills").iterrows():
    ax.annotate(row["iso3"], (row["gdp_pc"], row["cum_bills"]),
                xytext=(5,5), textcoords="offset points", fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("GDP per capita (USD, log scale)")
ax.set_ylabel("Cumulative AI bills passed (2016-2023)")
ax.set_title("AI legislation vs income")
ax.grid(alpha=0.3)
plt.tight_layout();

In [ ]:
# Correlation (Spearman handles the log-ish relationship better)
merged[["gdp_pc","cum_bills"]].corr(method="spearman")

### 8.5 Your turn

1. Fetch another indicator (e.g. `IT.NET.USER.ZS` — internet users %) and merge with `policy`.
   Is AI legislation more about connectivity or raw income?
2. Export OECD.AI's policy-initiatives CSV from
   `https://oecd.ai/en/dashboards/policy-initiatives`, drop it into `data/`, and break
   down policy **types** (strategy / regulation / funding) per country.
3. Write a function `rank_table(df, value_col, n)` that returns the top-n rows plus
   a "Others (sum)" row, and use it here.